In [11]:
try:
    load_type = getArgument("load_type")
    if load_type is None or load_type.strip() == "":
        load_type = "full"
except:
    load_type = "full"

print(f"Load type: {load_type}")

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 19, Finished, Available, Finished, False)

Load type: full


In [4]:
spark.sql("""
    UPDATE ingestion_metadata
    SET last_load_date = '2024-01-31'
    WHERE target_table = 'FactSales'
""")
print("Reset xong")
spark.sql("SELECT * FROM ingestion_metadata").show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 9, Finished, Available, Finished, False)

Reset xong
+------------+----------------+--------------+
|target_table|watermark_column|last_load_date|
+------------+----------------+--------------+
|   FactSales|   last_modified|    2024-01-31|
+------------+----------------+--------------+



In [5]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df_stg_cust = spark.table("stg_customers")

w = Window.orderBy("customer_id")
df_dim_cust = df_stg_cust \
    .withColumn("customer_key", row_number().over(w)) \
    .select("customer_key", "customer_id", "customer_name", "city", "segment")

df_dim_cust.write.mode("overwrite").saveAsTable("DimCustomer")
print(f"DimCustomer: {df_dim_cust.count()} rows")
df_dim_cust.show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 10, Finished, Available, Finished, False)

DimCustomer: 5 rows
+------------+-----------+-------------+-------+---------+
|customer_key|customer_id|customer_name|   city|  segment|
+------------+-----------+-------------+-------+---------+
|           1|       C001| Nguyen Van A|  HANOI|   Retail|
|           2|       C002|   Tran Thi B|   HCMC|Wholesale|
|           3|       C003|     Le Van C|DA NANG|   Retail|
|           4|       C004|   Pham Thi D|  HANOI|   Online|
|           5|       C005|  Hoang Van E|   HCMC|   Retail|
+------------+-----------+-------------+-------+---------+



In [6]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df_stg_prod = spark.table("stg_products")

w2 = Window.orderBy("product_id")
df_dim_prod = df_stg_prod \
    .withColumn("product_key", row_number().over(w2)) \
    .select("product_key", "product_id", "product_name", "category", "unit_price")

df_dim_prod.write.mode("overwrite").saveAsTable("DimProduct")
print(f"DimProduct: {df_dim_prod.count()} rows")
df_dim_prod.show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 12, Finished, Available, Finished, False)

DimProduct: 5 rows
+-----------+----------+-------------------+-----------+----------+
|product_key|product_id|       product_name|   category|unit_price|
+-----------+----------+-------------------+-----------+----------+
|          1|      P001|        Laptop Dell|Electronics|     1.5E7|
|          2|      P002|     Mouse Logitech|Electronics|  350000.0|
|          3|      P003|         Desk Chair|  Furniture| 2500000.0|
|          4|      P004|    Monitor Samsung|Electronics| 5000000.0|
|          5|      P005|Keyboard Mechanical|Electronics|  800000.0|
+-----------+----------+-------------------+-----------+----------+



In [7]:
from pyspark.sql.functions import col, to_date, monotonically_increasing_id
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df_stg_sales = spark.table("stg_sales")
df_dim_cust  = spark.table("DimCustomer")
df_dim_prod  = spark.table("DimProduct")

# ── Đọc last_load_date từ database ──────────────────────
meta = spark.sql("""
    SELECT last_load_date 
    FROM ingestion_metadata 
    WHERE target_table = 'FactSales'
""").collect()

last_load_date = str(meta[0]["last_load_date"])
print(f"Last load date từ DB: {last_load_date}")

# ── Chọn dữ liệu theo load_type ─────────────────────────
if load_type == "full":
    df_source = df_stg_sales
    write_mode = "overwrite"
    print(">> FULL LOAD: load toàn bộ dữ liệu")
else:
    df_source = df_stg_sales.filter(
        col("last_modified") > last_load_date
    )
    write_mode = "append"
    print(f">> INCREMENTAL LOAD: chỉ lấy dữ liệu sau {last_load_date}")

print(f"Số dòng cần load: {df_source.count()}")

# ── Transform ────────────────────────────────────────────
df_fact = df_source \
    .join(df_dim_cust.select("customer_key", "customer_id"), "customer_id", "left") \
    .join(df_dim_prod.select("product_key", "product_id"), "product_id", "left") \
    .withColumn("sales_key", monotonically_increasing_id()) \
    .select(
        col("sales_key"),
        col("order_id"),
        col("order_date"),
        col("customer_key"),
        col("product_key"),
        col("quantity"),
        col("line_amount").alias("revenue")
    )

# ── Ghi vào FactSales ────────────────────────────────────
df_fact.write.mode(write_mode).saveAsTable("FactSales")
print(f"FactSales {write_mode}: {df_fact.count()} rows")
df_fact.show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 14, Finished, Available, Finished, False)

Last load date từ DB: 2024-01-31
>> INCREMENTAL LOAD: chỉ lấy dữ liệu sau 2024-01-31
Số dòng cần load: 2
FactSales append: 2 rows
+---------+--------+----------+------------+-----------+--------+---------+
|sales_key|order_id|order_date|customer_key|product_key|quantity|  revenue|
+---------+--------+----------+------------+-----------+--------+---------+
|        0|  ORD007|2024-02-05|           2|          4|       2|    1.0E7|
|        1|  ORD006|2024-02-01|           1|          3|       1|2500000.0|
+---------+--------+----------+------------+-----------+--------+---------+



In [8]:
from datetime import date

# Lấy ngày max của last_modified vừa load
max_date = df_source.agg({"last_modified": "max"}).collect()[0][0]

if max_date:
    spark.sql(f"""
        UPDATE ingestion_metadata
        SET last_load_date = '{max_date}'
        WHERE target_table = 'FactSales'
    """)
    print(f"Đã cập nhật last_load_date = {max_date}")
else:
    print("Không có dữ liệu mới, giữ nguyên last_load_date")

# Kiểm tra lại
spark.sql("SELECT * FROM ingestion_metadata").show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 16, Finished, Available, Finished, False)

Đã cập nhật last_load_date = 2024-02-05
+------------+----------------+--------------+
|target_table|watermark_column|last_load_date|
+------------+----------------+--------------+
|   FactSales|   last_modified|    2024-02-05|
+------------+----------------+--------------+



In [10]:
spark.sql("SELECT COUNT(*) as total FROM dbo.factsales").show()

StatementMeta(, de83571a-d43f-468e-8412-56093dad5e08, 18, Finished, Available, Finished, False)

+-----+
|total|
+-----+
|   10|
+-----+

